# Model Adaptation Lab — Validation Walkthrough

This notebook runs the repository's **platform-independent validation path** on a fresh
Colab runtime.

**What it does:** validates the dataset contract and family-disjoint splits (v1 and v2),
runs the deterministic non-LLM baseline, and recomputes the committed evidence manifest
without model weights.

**What it does *not* do:** it does not train, convert, or evaluate the MLX model. That path
is Apple-Silicon-specific and requires base weights that are not in the checkout. This is a
validation notebook, not the full model-adaptation experiment.


## 1. Runtime check

In [ ]:
import platform
import shutil
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("rustc on PATH:", shutil.which("rustc") is not None)


## 2. Clone the repository

In [ ]:
import os
import subprocess

REPO_DIR = "model-adaptation-lab"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/rustfuture/model-adaptation-lab.git", REPO_DIR,
    ])

os.chdir(REPO_DIR)
print("working directory:", os.getcwd())


## 3. Validate the dataset contract (v1 and v2)

In [ ]:
def run(*args):
    result = subprocess.run(list(args), capture_output=True, text=True, check=True)
    print(result.stdout.strip())
    return result.stdout


print("== v1 dataset ==")
run(sys.executable, "scripts/validate_dataset.py")
print("\n== v2 dataset ==")
run(sys.executable, "scripts/validate_dataset_v2.py")


## 4. Deterministic non-LLM baseline

In [ ]:
print("== deterministic baseline (plumbing lower bound) ==")
run(sys.executable, "scripts/baseline.py")


## 5. Recompute the committed evidence manifest

`verify_evidence.py` reads only the authored dataset and tracked raw outputs. It never loads
model weights, so it runs anywhere. A byte-identical manifest proves the checked-in evidence
matches its sources.


In [ ]:
import tempfile

manifest_path = os.path.join(tempfile.mkdtemp(prefix="mal_"), "evidence-metadata.json")
run(sys.executable, "scripts/verify_evidence.py", "--write-manifest", manifest_path)

diff = subprocess.run(
    ["diff", "-u", "evidence/metadata.json", manifest_path],
    capture_output=True, text=True,
)
print("manifest diff exit code:", diff.returncode)
if diff.stdout:
    print(diff.stdout)
if diff.returncode != 0:
    raise SystemExit("evidence manifest does not match the committed file")
print("evidence manifest is reproducible from tracked sources")


## 6. Contract and provenance tests

In [ ]:
run(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_dataset_contract.py")
run(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_v2_provenance.py")


## 7. Optional: Rust snippet compile checks

These compile all 12 authored snippets and require the declared rustc diagnostic. They need
`rustc` on `PATH`; Colab does not ship it, so this cell reports the skip explicitly rather
than silently changing scope. Install a Rust toolchain if you want to run them.


In [ ]:
if shutil.which("rustc") is None:
    print("skipped: rustc is not on PATH (install a Rust toolchain to run the compile checks)")
else:
    print("== v1 snippets ==")
    run(sys.executable, "scripts/validate_rustc_snippets.py")
    print("\n== v2 snippets ==")
    run(sys.executable, "scripts/validate_rustc_snippets_v2.py")


## 8. How to read this

- These checks establish the **dataset contract**, the **deterministic baseline**, and the
  **integrity of the recorded evidence**.
- They do **not** establish behavioral or semantic correctness of the model outputs, and
  they do **not** reproduce MLX training.
- The historical result is deliberately negative: the recorded adapter did not improve the
  held-out keyword proxy. See `reports/negative-result.md` and `evidence/metadata.json`.
